# Análisis de importancia — modelo 9103

Objetivo: identificar sobre qué variables base vale la pena calcular medias móviles / tendencias en el 9105.

Procedimiento: entrenar UN modelo con los HP ganadores del 9103 sobre `dfinal_train`, extraer `lgb.importance()`, y agrupar variables por familia (quitando sufijos `_lag1/_lag2/_delta1/_delta2`) para identificar las **variables base más informativas**.

**Requisito:** este notebook debe correr en el kernel del 9103 (o 9104) con `dfinal_train`, `campos_buenos` y `dataset` en memoria. Si perdiste esa sesión, correr primero el 9103 hasta la celda del training strategy.

In [ ]:
# --- Sanity check: objetos necesarios en memoria ---
require("data.table")
require("lightgbm")

obligatorios <- c("dfinal_train", "campos_buenos")
faltantes <- obligatorios[!sapply(obligatorios, exists)]
if (length(faltantes) > 0) {
  stop("Faltan objetos: ", paste(faltantes, collapse = ", "),
       ". Necesito correr el 9103 hasta training strategy antes.")
}

cat("dfinal_train: filas=", dfinal_train$dim()[1], " cols=", length(campos_buenos), "\n")

In [ ]:
# --- Entreno UN modelo con los HP del 9103 para obtener importancia ---
# Uso una sola semilla, alcanza para el análisis de importancia

param_9103 <- list(
  objective = "binary",
  metric = "auc",
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = 804043,
  boosting = "gbdt",
  is_unbalance = FALSE,
  scale_pos_weight = 1.0,
  max_depth = -1,
  lambda_l1 = 0,
  lambda_l2 = 0,
  bagging_fraction = 1.0,
  bagging_freq = 0,
  min_gain_to_split = 0,
  min_data_in_leaf = 0,
  max_bin = 31,
  # HP ganadores del 9103
  learning_rate = 0.01318485,
  num_leaves = 256L,
  feature_fraction = 0.4214931,
  min_sum_hessian_in_leaf = 23.99688,
  num_iterations = 347L
)

cat("Entrenando modelo (esperado ~2-4 min)...\n")
t0 <- Sys.time()
modelo <- lgb.train(data = dfinal_train, param = param_9103, verbose = -100)
cat("Listo. Duracion:", round(as.numeric(difftime(Sys.time(), t0, units="mins")), 1), "min\n")

In [ ]:
# --- Importancia de variables ---
imp <- as.data.table(lgb.importance(modelo))
cat("Variables usadas por el modelo:", nrow(imp), "/", length(campos_buenos), "\n")
cat("(el resto tienen Gain = 0 y el modelo las ignoro)\n\n")

# top 30 individuales
cat("=== TOP 30 variables individuales por Gain ===\n")
print(imp[1:30, .(Feature, Gain = round(Gain, 4), Cover = round(Cover, 4), Frequency)])

# guardo la tabla completa a disco
fwrite(imp, "importancia_9103.txt", sep="\t")
cat("\nGuardado en importancia_9103.txt\n")

In [ ]:
# --- Agrupo por FAMILIA de variable ---
# Una variable como m_saldo_lag1 pertenece a la familia 'm_saldo'.
# Sumar la importancia por familia dice cuánto aporta la INFORMACIÓN de esa
# variable base considerando todas sus versiones (actual, lags y deltas).

# quito sufijos _lag1/_lag2/_delta1/_delta2 para identificar la variable base
imp[, familia := gsub("_(lag|delta)[0-9]+$", "", Feature)]

# agrego una marca del sufijo, para ver cual version aporta más
imp[, tipo := "base"]
imp[grepl("_lag1$", Feature), tipo := "lag1"]
imp[grepl("_lag2$", Feature), tipo := "lag2"]
imp[grepl("_delta1$", Feature), tipo := "delta1"]
imp[grepl("_delta2$", Feature), tipo := "delta2"]

# ranking por familia (suma de Gain de todas las versiones)
familia_rank <- imp[, .(
  gain_total = sum(Gain),
  n_versiones = .N,
  tipos = paste(sort(unique(tipo)), collapse = ",")
), by = familia][order(-gain_total)]

cat("=== TOP 30 FAMILIAS por Gain total (base + lags + deltas) ===\n")
print(familia_rank[1:30])

fwrite(familia_rank, "familias_importancia_9103.txt", sep="\t")
cat("\nGuardado en familias_importancia_9103.txt\n")

In [ ]:
# --- Analisis: ¿los deltas / lags aportan más que la variable base? ---
# Si los lags/deltas de una familia tienen MUCHO Gain, quiere decir que la
# INFORMACION TEMPORAL es lo importante para esa variable → buen candidato
# para agregarle medias moviles / tendencias en el 9105.

aporte_por_tipo <- imp[, .(gain = sum(Gain)), by = tipo]
aporte_por_tipo[, pct := round(100 * gain / sum(gain), 1)]
cat("=== ¿Cuánto aporta cada TIPO de feature al modelo? ===\n")
print(aporte_por_tipo[order(-gain)])

cat("\n=== Familias donde la INFORMACION TEMPORAL es dominante ===\n")
cat("(los lags+deltas aportan >70% del Gain de la familia)\n\n")

aporte_familia_por_tipo <- dcast(
  imp[, .(gain = sum(Gain)), by = .(familia, tipo)],
  familia ~ tipo, value.var = "gain", fill = 0
)

aporte_familia_por_tipo[, gain_total := rowSums(.SD, na.rm=TRUE),
                        .SDcols = setdiff(colnames(aporte_familia_por_tipo), "familia")]

cols_temporal <- intersect(c("lag1", "lag2", "delta1", "delta2"),
                            colnames(aporte_familia_por_tipo))
aporte_familia_por_tipo[, gain_temporal := rowSums(.SD, na.rm=TRUE),
                        .SDcols = cols_temporal]
aporte_familia_por_tipo[, pct_temporal := round(100 * gain_temporal / gain_total, 1)]

# Filtro: familias con Gain total significativo (>= mediano del top 50) y >70% de peso temporal
top_families <- familia_rank[1:50, familia]
candidatos_movil <- aporte_familia_por_tipo[
  familia %in% top_families & pct_temporal >= 70
][order(-gain_total)]

print(candidatos_movil[, .(familia, gain_total = round(gain_total, 4),
                            pct_temporal, base = round(base, 4),
                            lag1 = round(lag1, 4), delta1 = round(delta1, 4))])

cat("\n=== RECOMENDACION ===\n")
cat("Las familias arriba son las mejores candidatas para calcular medias moviles\n")
cat("y tendencias en el 9105. Su señal esta en la evolucion temporal, no en el\n")
cat("valor puntual.\n")

fwrite(candidatos_movil, "candidatos_moviles_9105.txt", sep="\t")